In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv


In [1]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv


In [3]:

import pandas as pd

path = "/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(path)

print("Размер датасета:", df.shape)

print("\nПервые 5 строк:")
display(df.head())

print("\nТипы данных:")
print(df.dtypes)

print("\nПропущенные значения:")
print(df.isnull().sum())

print("\nРаспределение целевой переменной:")
print(df["Churn"].value_counts())
print("\nДоля классов, %:")
print((df["Churn"].value_counts(normalize=True) * 100).round(2))


Размер датасета: (7043, 21)

Первые 5 строк:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes



Типы данных:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

Пропущенные значения:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
Paperless

In [4]:

print("Примеры значений TotalCharges:")
print(df["TotalCharges"].unique()[:20])

print("\nКоличество пустых строк:")
print((df["TotalCharges"].astype(str).str.strip() == "").sum())

print("\nКоличество уникальных Customer ID:")
print(df["customerID"].nunique())


Примеры значений TotalCharges:
['29.85' '1889.5' '108.15' '1840.75' '151.65' '820.5' '1949.4' '301.9'
 '3046.05' '3487.95' '587.45' '326.8' '5681.1' '5036.3' '2686.05'
 '7895.15' '1022.95' '7382.25' '528.35' '1862.9']

Количество пустых строк:
11

Количество уникальных Customer ID:
7043


In [5]:
# Показываем строки с пустым TotalCharges
empty_charges = df[
    df["TotalCharges"].astype(str).str.strip() == ""
]

display(empty_charges[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]])

,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


In [6]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].astype(str).str.strip(),
    errors="coerce"
)

print("Тип после преобразования:", df["TotalCharges"].dtype)
print("Пропуски после преобразования:", df["TotalCharges"].isna().sum())

Тип после преобразования: float64
Пропуски после преобразования: 11


In [7]:

# 1. Отток в зависимости от типа контракта
contract_churn = pd.crosstab(
    df["Contract"],
    df["Churn"],
    normalize="index"
) * 100

print("Доля ушедших и оставшихся по типу контракта, %:")
display(contract_churn.round(2))

# 2. Сравнение срока обслуживания
print("\nСрок обслуживания по группам Churn:")
display(
    df.groupby("Churn")["tenure"]
      .agg(["count", "mean", "median", "min", "max"])
      .round(2)
)

# 3. Сравнение ежемесячной платы
print("\nЕжемесячная плата по группам Churn:")
display(
    df.groupby("Churn")["MonthlyCharges"]
      .agg(["mean", "median"])
      .round(2)
)


Доля ушедших и оставшихся по типу контракта, %:


Churn,No,Yes
Contract,,
Month-to-month,57.29,42.71
One year,88.73,11.27
Two year,97.17,2.83



Срок обслуживания по группам Churn:


,count,mean,median,min,max
Churn,,,,,
No,5174,37.57,38.0,0,72
Yes,1869,17.98,10.0,1,72



Ежемесячная плата по группам Churn:


,mean,median
Churn,,
No,61.27,64.43
Yes,74.44,79.65


In [8]:

# Доля оттока одновременно по контракту и сроку обслуживания
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 6, 12, 24, 48, 72],
    labels=["0–6", "7–12", "13–24", "25–48", "49–72"]
)

churn_by_contract_tenure = pd.pivot_table(
    df,
    values="Churn",
    index="TenureGroup",
    columns="Contract",
    aggfunc=lambda x: (x == "Yes").mean() * 100,
    observed=False
)

print("Доля оттока по сроку обслуживания и типу контракта, %:")
display(churn_by_contract_tenure.round(1))



Доля оттока по сроку обслуживания и типу контракта, %:


Contract,Month-to-month,One year,Two year
TenureGroup,,,
0–6,55.2,10.3,0.0
7–12,42.0,10.6,0.0
13–24,37.7,8.1,0.0
25–48,32.9,10.6,2.2
49–72,26.0,12.9,3.3


In [9]:
support_churn = pd.crosstab(
    df["TechSupport"],
    df["Churn"],
    normalize="index"
) * 100

display(support_churn.round(2))

Churn,No,Yes
TechSupport,,
No,58.36,41.64
No internet service,92.60,7.40
Yes,84.83,15.17


In [10]:
support_contract_churn = pd.pivot_table(
    df,
    values="Churn",
    index="TechSupport",
    columns="Contract",
    aggfunc=lambda x: (x == "Yes").mean() * 100
)

display(support_contract_churn.round(2))

Contract,Month-to-month,One year,Two year
TechSupport,,,
No,50.37,14.72,5.93
No internet service,18.89,2.47,0.78
Yes,30.70,13.59,3.53


In [11]:
support_contract_tenure = pd.pivot_table(
    df,
    values="Churn",
    index=["TenureGroup", "TechSupport"],
    columns="Contract",
    aggfunc=lambda x: (x == "Yes").mean() * 100,
    observed=False
)

display(support_contract_tenure.round(1))

Contract                         Month-to-month  One year  Two year
TenureGroup TechSupport                                            
0–6         No                             65.7      33.3       0.0
            No internet service            25.4       6.1       0.0
            Yes                            46.6      33.3       0.0
7–12        No                             49.6      22.7       0.0
            No internet service            13.5       4.8       0.0
            Yes                            36.8       9.5       0.0
13–24       No                             44.2      14.0       0.0
            No internet service            10.0       1.1       0.0
            Yes                            29.9      15.4       0.0
25–48       No                             38.6      11.8       4.8
            No internet service             3.7       1.6       1.2
            Yes                            25.1      15.0       3.2
49–72       No                             31.0      16.0       6.3
            No internet service            50.0       2.8       0.8
            Yes                            10.6      12.3       3.8

In [12]:

group_stats = df.pivot_table(
    index=["TenureGroup", "TechSupport"],
    columns="Contract",
    values="Churn",
    aggfunc=[
        "count",
        lambda x: (x == "Yes").mean() * 100
    ],
    observed=False
)

display(group_stats.round(1))


count                    \
Contract                        Month-to-month One year Two year   
TenureGroup TechSupport                                            
0–6         No                             966        3        1   
            No internet service            299       33       19   
            Yes                            148        3        9   
7–12        No                             397       22        1   
            No internet service             89       42       31   
            Yes                             95       21        7   
13–24       No                             513       50        6   
            No internet service             80       95       64   
            Yes                            144       52       20   
25–48       No                             549      195       21   
            No internet service             54      123      160   
            Yes                            199      200       93   
49–72       No                             255      287      207   
            No internet service              2       71      364   
            Yes                             85      276      692   

                                      <lambda>                    
Contract                        Month-to-month One year Two year  
TenureGroup TechSupport                                           
0–6         No                            65.7     33.3      0.0  
            No internet service           25.4      6.1      0.0  
            Yes                           46.6     33.3      0.0  
7–12        No                            49.6     22.7      0.0  
            No internet service           13.5      4.8      0.0  
            Yes                           36.8      9.5      0.0  
13–24       No                            44.2     14.0      0.0  
            No internet service           10.0      1.1      0.0  
            Yes                           29.9     15.4      0.0  
25–48       No                            38.6     11.8      4.8  
            No internet service            3.7      1.6      1.2  
            Yes                           25.1     15.0      3.2  
49–72       No                            31.0     16.0      6.3  
            No internet service           50.0      2.8      0.8  
            Yes                           10.6     12.3      3.8

In [13]:

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Убираем идентификатор и временную исследовательскую группу.
# Tenure оставляем в исходном числовом виде.
X = df.drop(columns=["customerID", "Churn", "TenureGroup"])
y = (df["Churn"] == "Yes").astype(int)

# Определяем типы признаков
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()

# Числовые признаки: заполняем пропуски медианой и масштабируем.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Категориальные признаки: заполняем пропуски и кодируем категории.
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Отделяем данные для честной проверки модели.
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model.fit(X_train, y_train)

# Вероятность оттока и итоговая классификация.
probabilities = model.predict_proba(X_val)[:, 1]
predictions = (probabilities >= 0.5).astype(int)

print("Accuracy:", round(accuracy_score(y_val, predictions), 4))
print("Precision:", round(precision_score(y_val, predictions), 4))
print("Recall:", round(recall_score(y_val, predictions), 4))
print("F1:", round(f1_score(y_val, predictions), 4))
print("ROC-AUC:", round(roc_auc_score(y_val, probabilities), 4))

print("\nМатрица ошибок:")
print(confusion_matrix(y_val, predictions))


Accuracy: 0.8055
Precision: 0.6572
Recall: 0.5588
F1: 0.604
ROC-AUC: 0.8419

Матрица ошибок:
[[926 109]
 [165 209]]


In [14]:

# Создаём таблицу для анализа ошибок
error_analysis = X_val.copy()

error_analysis["Actual"] = y_val
error_analysis["Predicted"] = predictions
error_analysis["ChurnProbability"] = probabilities

# Категоризируем результаты
error_analysis["ErrorType"] = "Correct"

error_analysis.loc[
    (error_analysis["Actual"] == 1) &
    (error_analysis["Predicted"] == 0),
    "ErrorType"
] = "Missed churn"

error_analysis.loc[
    (error_analysis["Actual"] == 0) &
    (error_analysis["Predicted"] == 1),
    "ErrorType"
] = "False alarm"

print("Количество по типам результата:")
print(error_analysis["ErrorType"].value_counts())

print("\nСредняя вероятность ухода по типам результата:")
display(
    error_analysis.groupby("ErrorType")["ChurnProbability"]
    .agg(["count", "mean", "median"])
    .round(3)
)


Количество по типам результата:
ErrorType
Correct         1135
Missed churn     165
False alarm      109
Name: count, dtype: int64

Средняя вероятность ухода по типам результата:


,count,mean,median
ErrorType,,,
Correct,1135,0.232,0.125
False alarm,109,0.624,0.608
Missed churn,165,0.278,0.280


In [15]:

features_to_check = [
    "PaymentMethod",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "PaperlessBilling"
]

for feature in features_to_check:
    print(f"\n--- {feature} ---")

    display(
        pd.crosstab(
            error_analysis["ErrorType"],
            X_val[feature],
            normalize="index"
        ).round(3) * 100
    )



--- PaymentMethod ---


PaymentMethod,Bank transfer (automatic),Credit card (automatic),Electronic check,Mailed check
ErrorType,,,,
Correct,22.4,23.8,30.3,23.5
False alarm,7.3,9.2,69.7,13.8
Missed churn,23.0,17.6,32.7,26.7



--- InternetService ---


InternetService,DSL,Fiber optic,No
ErrorType,,,
Correct,35.5,39.2,25.3
False alarm,10.1,89.9,0.0
Missed churn,42.4,42.4,15.2



--- OnlineSecurity ---


...

OnlineBackup,No,No internet service,Yes
ErrorType,,,
Correct,42.4,25.3,32.3
False alarm,78.9,0.0,21.1
Missed churn,47.9,15.2,37.0



--- DeviceProtection ---


DeviceProtection,No,No internet service,Yes
ErrorType,,,
Correct,39.0,25.3,35.7
False alarm,81.7,0.0,18.3
Missed churn,55.2,15.2,29.7



--- PaperlessBilling ---


PaperlessBilling,No,Yes
ErrorType,,
Correct,42.6,57.4
False alarm,17.4,82.6
Missed churn,40.6,59.4


In [16]:

features_to_check = [
    "PaymentMethod",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "PaperlessBilling"
]

for feature in features_to_check:
    print(f"\n--- {feature} ---")

    result = df.groupby(feature, dropna=False).agg(
        customers=("Churn", "size"),
        churn_rate=("Churn", lambda x: (x == "Yes").mean() * 100)
    )

    display(result.round(2))



--- PaymentMethod ---


,customers,churn_rate
PaymentMethod,,
Bank transfer (automatic),1544,16.71
Credit card (automatic),1522,15.24
Electronic check,2365,45.29
Mailed check,1612,19.11



--- InternetService ---


,customers,churn_rate
InternetService,,
DSL,2421,18.96
Fiber optic,3096,41.89
No,1526,7.40



--- OnlineSecurity ---


...

,customers,churn_rate
OnlineBackup,,
No,3088,39.93
No internet service,1526,7.40
Yes,2429,21.53



--- DeviceProtection ---


,customers,churn_rate
DeviceProtection,,
No,3095,39.13
No internet service,1526,7.40
Yes,2422,22.50



--- PaperlessBilling ---


,customers,churn_rate
PaperlessBilling,,
No,2872,16.33
Yes,4171,33.57


In [17]:

contract_analysis = df.groupby("Contract").agg(
    customers=("customerID", "count"),
    avg_monthly_charges=("MonthlyCharges", "mean"),
    median_monthly_charges=("MonthlyCharges", "median"),
    avg_total_charges=("TotalCharges", "mean"),
    median_total_charges=("TotalCharges", "median"),
    churn_rate=("Churn", lambda x: (x == "Yes").mean() * 100)
)

display(contract_analysis.round(2))


,customers,avg_monthly_charges,median_monthly_charges,avg_total_charges,median_total_charges,churn_rate
Contract,,,,,,
Month-to-month,3875,66.40,73.25,1369.25,679.55,42.71
One year,1473,65.05,68.75,3034.68,2657.55,11.27
Two year,1695,60.77,64.35,3728.93,3623.95,2.83


In [18]:

df["ChargeGroup"] = pd.qcut(
    df["MonthlyCharges"],
    q=4,
    labels=["Низкая", "Средняя", "Выше средней", "Высокая"]
)

price_contract_analysis = df.pivot_table(
    index="ChargeGroup",
    columns="Contract",
    values="Churn",
    aggfunc=lambda x: (x == "Yes").mean() * 100,
    observed=False
)

print("Доля оттока по уровню ежемесячной платы и контракту, %:")
display(price_contract_analysis.round(2))


Доля оттока по уровню ежемесячной платы и контракту, %:


Contract,Month-to-month,One year,Two year
ChargeGroup,,,
Низкая,25.03,3.05,0.78
Средняя,35.62,8.63,2.45
Выше средней,53.28,11.07,1.55
Высокая,52.18,20.85,7.01


In [19]:

price_internet_contract = pd.pivot_table(
    df,
    index=["Contract", "InternetService"],
    columns="ChargeGroup",
    values="Churn",
    aggfunc=lambda x: (x == "Yes").mean() * 100,
    observed=False
)

print("Доля оттока по контракту, интернету и уровню платы, %:")
display(price_internet_contract.round(2))


Доля оттока по контракту, интернету и уровню платы, %:


ChargeGroup                     Низкая  Средняя  Выше средней  Высокая
Contract       InternetService                                        
Month-to-month DSL               41.21    31.58         16.22      NaN
               Fiber optic         NaN    59.75         55.78    52.18
               No                18.89      NaN           NaN      NaN
One year       DSL               10.00     8.63         10.84     0.00
               Fiber optic         NaN      NaN         11.46    20.99
               No                 2.47      NaN           NaN      NaN
Two year       DSL                0.00     2.46          1.38     2.13
               Fiber optic         NaN     0.00          3.12     7.59
               No                 0.78      NaN           NaN      NaN

In [20]:

group_sizes = pd.pivot_table(
    df,
    index=["Contract", "InternetService"],
    columns="ChargeGroup",
    values="Churn",
    aggfunc="count",
    observed=False
)

print("Количество клиентов в каждой группе:")
display(group_sizes)


Количество клиентов в каждой группе:


ChargeGroup                     Низкая  Средняя  Выше средней  Высокая
Contract       InternetService                                        
Month-to-month DSL                 199      950            74        0
               Fiber optic           0      159          1099      870
               No                  524        0             0        0
One year       DSL                  30      371           166        3
               Fiber optic           0        0            96      443
               No                  364        0             0        0
Two year       DSL                   7      284           290       47
               Fiber optic           0        2            32      395
               No                  638        0             0        0

In [21]:

price_summary = df.groupby(
    ["Contract", "InternetService"],
    observed=False
).agg(
    customers=("MonthlyCharges", "size"),
    mean_price=("MonthlyCharges", "mean"),
    median_price=("MonthlyCharges", "median"),
    min_price=("MonthlyCharges", "min"),
    max_price=("MonthlyCharges", "max"),
    churn_rate=("Churn", lambda x: (x == "Yes").mean() * 100)
)

display(price_summary.round(2))


customers  mean_price  median_price  \
Contract       InternetService                                        
Month-to-month DSL                   1223       50.22         50.20   
               Fiber optic           2128       87.02         86.00   
               No                     524       20.41         20.05   
One year       DSL                    570       61.40         61.05   
               Fiber optic            539       98.78        100.15   
               No                     364       20.82         20.10   
Two year       DSL                    628       70.46         73.30   
               Fiber optic            429      104.57        106.30   
               No                     638       21.78         20.35   

                                min_price  max_price  churn_rate  
Contract       InternetService                                    
Month-to-month DSL                  23.45      86.20       32.22  
               Fiber optic          67.75     117.45       54.61  
               No                   18.75      26.45       18.89  
One year       DSL                  24.10      90.60        9.30  
               Fiber optic          70.40     118.60       19.29  
               No                   18.25      26.90        2.47  
Two year       DSL                  29.60      94.80        1.91  
               Fiber optic          69.05     118.75        7.23  
               No                   18.40      26.50        0.78

In [22]:

tenure_analysis = df.pivot_table(
    index=["Contract", "InternetService"],
    columns="TenureGroup",
    values="Churn",
    aggfunc=lambda x: (x == "Yes").mean() * 100,
    observed=False
)

print("Доля оттока по контракту, интернету и стажу, %:")
display(tenure_analysis.round(2))


Доля оттока по контракту, интернету и стажу, %:


TenureGroup                        0–6   7–12  13–24  25–48  49–72
Contract       InternetService                                    
Month-to-month DSL               49.49  24.62  23.71  15.86  13.51
               Fiber optic       74.15  61.95  50.59  43.38  29.32
               No                25.42  13.48  10.00   3.70  50.00
One year       DSL               20.00  16.22  13.92   9.13   6.25
               Fiber optic      100.00  16.67  17.39  20.13  18.87
               No                 6.06   4.76   1.05   1.63   2.82
Two year       DSL                0.00   0.00   0.00   1.27   2.17
               Fiber optic         NaN    NaN   0.00   8.57   7.12
               No                 0.00   0.00   0.00   1.25   0.82

In [23]:

tenure_counts = df.pivot_table(
    index=["Contract", "InternetService"],
    columns="TenureGroup",
    values="Churn",
    aggfunc="count",
    observed=False
)

print("Количество клиентов по контракту, интернету и стажу:")
display(tenure_counts)


Количество клиентов по контракту, интернету и стажу:


TenureGroup                     0–6  7–12  13–24  25–48  49–72
Contract       InternetService                                
Month-to-month DSL              495   195    232    227     74
               Fiber optic      619   297    425    521    266
               No               299    89     80     54      2
One year       DSL                5    37     79    241    208
               Fiber optic        1     6     23    154    355
               No                33    42     95    123     71
Two year       DSL               10     8     25     79    506
               Fiber optic        0     0      1     35    393
               No                19    31     64    160    364

In [24]:

security_analysis = df.pivot_table(
    index=["Contract", "InternetService", "OnlineSecurity"],
    values="Churn",
    aggfunc=[
        "count",
        lambda x: (x == "Yes").mean() * 100
    ],
    observed=False
)

security_analysis.columns = ["customers", "churn_rate"]
security_analysis = security_analysis.reset_index()

print("Количество клиентов и отток по OnlineSecurity:")
display(security_analysis.round(2))


Количество клиентов и отток по OnlineSecurity:


,Contract,InternetService,OnlineSecurity,customers,churn_rate
0,Month-to-month,DSL,No,857,36.41
1,Month-to-month,DSL,Yes,366,22.40
2,Month-to-month,Fiber optic,No,1774,58.12
3,Month-to-month,Fiber optic,Yes,354,37.01
4,Month-to-month,No,No internet service,524,18.89
5,One year,DSL,No,240,12.92
6,One year,DSL,Yes,330,6.67
7,One year,Fiber optic,No,317,20.82
8,One year,Fiber optic,Yes,222,17.12
9,One year,No,No internet service,364,2.47


In [25]:

support_analysis = df.pivot_table(
    index=["Contract", "InternetService", "TechSupport"],
    values="Churn",
    aggfunc=[
        "count",
        lambda x: (x == "Yes").mean() * 100
    ],
    observed=False
)

support_analysis.columns = ["customers", "churn_rate"]
support_analysis = support_analysis.reset_index()

print("Количество клиентов и отток по технической поддержке:")
display(support_analysis.round(2))


Количество клиентов и отток по технической поддержке:


,Contract,InternetService,TechSupport,customers,churn_rate
0,Month-to-month,DSL,No,884,35.86
1,Month-to-month,DSL,Yes,339,22.71
2,Month-to-month,Fiber optic,No,1796,57.52
3,Month-to-month,Fiber optic,Yes,332,38.86
4,Month-to-month,No,No internet service,524,18.89
5,One year,DSL,No,244,9.84
6,One year,DSL,Yes,326,8.90
7,One year,Fiber optic,No,313,18.53
8,One year,Fiber optic,Yes,226,20.35
9,One year,No,No internet service,364,2.47


In [26]:

import pandas as pd
import numpy as np

# Получаем названия признаков после обработки данных
preprocessor = model.named_steps["preprocessor"]
classifier = model.named_steps["classifier"]

feature_names = preprocessor.get_feature_names_out()
coefficients = classifier.coef_[0]

coef_table = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "OddsRatio": np.exp(coefficients)
})

coef_table["AbsCoefficient"] = coef_table["Coefficient"].abs()

display(
    coef_table.sort_values(
        "AbsCoefficient",
        ascending=False
    ).drop(columns="AbsCoefficient").round(3)
)


,Feature,Coefficient,OddsRatio
1,num__tenure,-1.241,0.289
38,cat__Contract_Two year,-0.772,0.462
15,cat__InternetService_DSL,-0.652,0.521
16,cat__InternetService_Fiber optic,0.640,1.897
2,num__MonthlyCharges,-0.596,0.551
36,cat__Contract_Month-to-month,0.579,1.785
3,num__TotalCharges,0.517,1.676
39,cat__PaperlessBilling_No,-0.343,0.709
25,cat__DeviceProtection_No internet service,-0.301,0.740
17,cat__InternetService_No,-0.301,0.740


In [2]:

numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

correlation = df[numeric_features].corr()

print("Корреляционная матрица:")
display(correlation.round(3))


NameError: name 'df' is not defined

In [3]:
print("df" in globals())

False


In [4]:
import pandas as pd

df = pd.read_csv(
    "/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

print(df.shape)
display(df.head())

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

correlation = df[numeric_features].corr()

print("Корреляционная матрица:")
display(correlation.round(3))

Корреляционная матрица:


,tenure,MonthlyCharges,TotalCharges
tenure,1.000,0.248,0.826
MonthlyCharges,0.248,1.000,0.651
TotalCharges,0.826,0.651,1.000


In [6]:

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import pandas as pd
import numpy as np

# Модель A: все признаки, включая TotalCharges
model_A = clone(model)
model_A.fit(X_train, y_train)

# Модель B: убираем только TotalCharges
X_train_B = X_train.drop(columns=["TotalCharges"])
X_val_B = X_val.drop(columns=["TotalCharges"])

model_B = clone(model)
model_B.fit(X_train_B, y_train)

# Сравниваем прогнозы на одной проверочной выборке
prob_A = model_A.predict_proba(X_val)[:, 1]
prob_B = model_B.predict_proba(X_val_B)[:, 1]

pred_A = (prob_A >= 0.5).astype(int)
pred_B = (prob_B >= 0.5).astype(int)

results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
    "With_TotalCharges": [
        accuracy_score(y_val, pred_A),
        precision_score(y_val, pred_A),
        recall_score(y_val, pred_A),
        f1_score(y_val, pred_A),
        roc_auc_score(y_val, prob_A)
    ],
    "Without_TotalCharges": [
        accuracy_score(y_val, pred_B),
        precision_score(y_val, pred_B),
        recall_score(y_val, pred_B),
        f1_score(y_val, pred_B),
        roc_auc_score(y_val, prob_B)
    ]
})

display(results.round(4))

# Сравниваем коэффициенты MonthlyCharges
def get_coefficient_table(fitted_model):
    preprocessor = fitted_model.named_steps["preprocessor"]
    classifier = fitted_model.named_steps["classifier"]
    names = preprocessor.get_feature_names_out()
    coefs = classifier.coef_[0]

    return pd.DataFrame({
        "Feature": names,
        "Coefficient": coefs,
        "OddsRatio": np.exp(coefs)
    })

coef_A = get_coefficient_table(model_A)
coef_B = get_coefficient_table(model_B)

print("MonthlyCharges — модель A:")
display(coef_A[coef_A["Feature"] == "num__MonthlyCharges"].round(4))

print("MonthlyCharges — модель B:")
display(coef_B[coef_B["Feature"] == "num__MonthlyCharges"].round(4))


NameError: name 'model' is not defined

In [7]:

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Восстанавливаем целевую переменную и признаки
X = df.drop(columns=["customerID", "Churn", "TenureGroup"], errors="ignore")
y = (df["Churn"] == "Yes").astype(int)

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Используем то же разбиение, что и раньше
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Модель и выборки восстановлены.")
print("Обучающая выборка:", X_train.shape)
print("Проверочная выборка:", X_val.shape)


Модель и выборки восстановлены.
Обучающая выборка: (5634, 19)
Проверочная выборка: (1409, 19)


In [8]:

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import pandas as pd
import numpy as np

# Модель A: все признаки, включая TotalCharges
model_A = clone(model)
model_A.fit(X_train, y_train)

# Модель B: убираем только TotalCharges
X_train_B = X_train.drop(columns=["TotalCharges"])
X_val_B = X_val.drop(columns=["TotalCharges"])

model_B = clone(model)
model_B.fit(X_train_B, y_train)

# Сравниваем прогнозы на одной проверочной выборке
prob_A = model_A.predict_proba(X_val)[:, 1]
prob_B = model_B.predict_proba(X_val_B)[:, 1]

pred_A = (prob_A >= 0.5).astype(int)
pred_B = (prob_B >= 0.5).astype(int)

results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
    "With_TotalCharges": [
        accuracy_score(y_val, pred_A),
        precision_score(y_val, pred_A),
        recall_score(y_val, pred_A),
        f1_score(y_val, pred_A),
        roc_auc_score(y_val, prob_A)
    ],
    "Without_TotalCharges": [
        accuracy_score(y_val, pred_B),
        precision_score(y_val, pred_B),
        recall_score(y_val, pred_B),
        f1_score(y_val, pred_B),
        roc_auc_score(y_val, prob_B)
    ]
})

display(results.round(4))

# Сравниваем коэффициенты MonthlyCharges
def get_coefficient_table(fitted_model):
    preprocessor = fitted_model.named_steps["preprocessor"]
    classifier = fitted_model.named_steps["classifier"]
    names = preprocessor.get_feature_names_out()
    coefs = classifier.coef_[0]

    return pd.DataFrame({
        "Feature": names,
        "Coefficient": coefs,
        "OddsRatio": np.exp(coefs)
    })

coef_A = get_coefficient_table(model_A)
coef_B = get_coefficient_table(model_B)

print("MonthlyCharges — модель A:")
display(coef_A[coef_A["Feature"] == "num__MonthlyCharges"].round(4))

print("MonthlyCharges — модель B:")
display(coef_B[coef_B["Feature"] == "num__MonthlyCharges"].round(4))


ValueError: A given column is not a column of the dataframe

In [9]:

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)
import pandas as pd
import numpy as np

# Модель A: с TotalCharges
model_A = clone(model)
model_A.fit(X_train, y_train)

# Модель B: без TotalCharges
X_train_B = X_train.drop(columns=["TotalCharges"])
X_val_B = X_val.drop(columns=["TotalCharges"])

model_B = clone(model)
model_B.fit(X_train_B, y_train)

# Сравниваем прогнозы на одной проверочной выборке
prob_A = model_A.predict_proba(X_val)[:, 1]
prob_B = model_B.predict_proba(X_val_B)[:, 1]

pred_A = (prob_A >= 0.5).astype(int)
pred_B = (prob_B >= 0.5).astype(int)

results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
    "With_TotalCharges": [
        accuracy_score(y_val, pred_A),
        precision_score(y_val, pred_A),
        recall_score(y_val, pred_A),
        f1_score(y_val, pred_A),
        roc_auc_score(y_val, prob_A)
    ],
    "Without_TotalCharges": [
        accuracy_score(y_val, pred_B),
        precision_score(y_val, pred_B),
        recall_score(y_val, pred_B),
        f1_score(y_val, pred_B),
        roc_auc_score(y_val, prob_B)
    ]
})

display(results.round(4))

# Сравниваем коэффициент MonthlyCharges
def get_coefficient_table(fitted_model):
    names = fitted_model.named_steps["preprocessor"].get_feature_names_out()
    coefs = fitted_model.named_steps["classifier"].coef_[0]

    return pd.DataFrame({
        "Feature": names,
        "Coefficient": coefs,
        "OddsRatio": np.exp(coefs)
    })

coef_A = get_coefficient_table(model_A)
coef_B = get_coefficient_table(model_B)

print("Модель A — с TotalCharges:")
display(coef_A[coef_A["Feature"] == "num__MonthlyCharges"].round(4))

print("Модель B — без TotalCharges:")
display(coef_B[coef_B["Feature"] == "num__MonthlyCharges"].round(4))


ValueError: A given column is not a column of the dataframe

In [10]:

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)
import pandas as pd
import numpy as np

# Модель A: с TotalCharges
model_A = clone(model)
model_A.fit(X_train, y_train)

# Модель B: без TotalCharges
X_train_B = X_train.drop(columns=["TotalCharges"])
X_val_B = X_val.drop(columns=["TotalCharges"])

numeric_cols_B = [
    col for col in numeric_cols
    if col != "TotalCharges"
]

categorical_cols_B = categorical_cols

numeric_pipeline_B = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline_B = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_B = ColumnTransformer([
    ("num", numeric_pipeline_B, numeric_cols_B),
    ("cat", categorical_pipeline_B, categorical_cols_B)
])

model_B = Pipeline([
    ("preprocessor", preprocessor_B),
    ("classifier", LogisticRegression(max_iter=1000))
])

model_B.fit(X_train_B, y_train)

# Сравниваем качество на одной проверочной выборке
prob_A = model_A.predict_proba(X_val)[:, 1]
prob_B = model_B.predict_proba(X_val_B)[:, 1]

pred_A = (prob_A >= 0.5).astype(int)
pred_B = (prob_B >= 0.5).astype(int)

results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
    "With_TotalCharges": [
        accuracy_score(y_val, pred_A),
        precision_score(y_val, pred_A),
        recall_score(y_val, pred_A),
        f1_score(y_val, pred_A),
        roc_auc_score(y_val, prob_A)
    ],
    "Without_TotalCharges": [
        accuracy_score(y_val, pred_B),
        precision_score(y_val, pred_B),
        recall_score(y_val, pred_B),
        f1_score(y_val, pred_B),
        roc_auc_score(y_val, prob_B)
    ]
})

print("Сравнение качества моделей:")
display(results.round(4))

def get_coefficient_table(fitted_model):
    names = fitted_model.named_steps["preprocessor"].get_feature_names_out()
    coefs = fitted_model.named_steps["classifier"].coef_[0]

    return pd.DataFrame({
        "Feature": names,
        "Coefficient": coefs,
        "OddsRatio": np.exp(coefs)
    })

coef_A = get_coefficient_table(model_A)
coef_B = get_coefficient_table(model_B)

print("MonthlyCharges — модель A:")
display(
    coef_A[coef_A["Feature"] == "num__MonthlyCharges"].round(4)
)

print("MonthlyCharges — модель B:")
display(
    coef_B[coef_B["Feature"] == "num__MonthlyCharges"].round(4)
)


Сравнение качества моделей:


,Metric,With_TotalCharges,Without_TotalCharges
0,Accuracy,0.8055,0.7977
1,Precision,0.6572,0.6378
2,Recall,0.5588,0.5508
3,F1,0.6040,0.5911
4,ROC-AUC,0.8419,0.8391


MonthlyCharges — модель A:


,Feature,Coefficient,OddsRatio
2,num__MonthlyCharges,-0.5964,0.5508


MonthlyCharges — модель B:


,Feature,Coefficient,OddsRatio
2,num__MonthlyCharges,-0.5021,0.6052


In [11]:

# Создаём четыре группы по ежемесячной плате
df["ChargeGroup"] = pd.qcut(
    df["MonthlyCharges"],
    q=4,
    labels=["Низкая", "Средняя", "Выше средней", "Высокая"]
)

# Считаем размер групп и долю оттока
price_interaction = df.pivot_table(
    index=["Contract", "InternetService"],
    columns="ChargeGroup",
    values="Churn",
    aggfunc=lambda x: (x == "Yes").mean() * 100,
    observed=False
)

print("Отток по цене внутри контрактов и типов интернета, %:")
display(price_interaction.round(2))


Отток по цене внутри контрактов и типов интернета, %:


ChargeGroup                     Низкая  Средняя  Выше средней  Высокая
Contract       InternetService                                        
Month-to-month DSL               41.21    31.58         16.22      NaN
               Fiber optic         NaN    59.75         55.78    52.18
               No                18.89      NaN           NaN      NaN
One year       DSL               10.00     8.63         10.84     0.00
               Fiber optic         NaN      NaN         11.46    20.99
               No                 2.47      NaN           NaN      NaN
Two year       DSL                0.00     2.46          1.38     2.13
               Fiber optic         NaN     0.00          3.12     7.59
               No                 0.78      NaN           NaN      NaN

In [1]:
from sklearn.pipeline import Pipeline

In [2]:

# Импортируем алгоритмы и метрики для сравнения моделей.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.base import clone
import pandas as pd

# Создаём три модели.
# max_iter — максимальное число итераций поиска решения для логистической регрессии.
# max_depth ограничивает глубину дерева, чтобы снизить риск переобучения.
# n_estimators — количество деревьев в случайном лесу.
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
}

results = []

for name, classifier in models.items():
    # clone(preprocessor) создаёт отдельный экземпляр обработки данных
    # для каждой модели, не смешивая их настройки.
    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", classifier)
    ])

    # Обучаем модель только на тренировочных данных.
    pipeline.fit(X_train, y_train)

    # Предсказываем класс клиента и вероятность его ухода.
    predictions = pipeline.predict(X_val)
    probabilities = pipeline.predict_proba(X_val)[:, 1]

    # Считаем метрики, чтобы сравнить качество моделей.
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, predictions),
        "Precision": precision_score(y_val, predictions, zero_division=0),
        "Recall": recall_score(y_val, predictions, zero_division=0),
        "F1": f1_score(y_val, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_val, probabilities)
    })

# Превращаем результаты в таблицу и сортируем по ROC-AUC.
results_df = pd.DataFrame(results).sort_values(
    by="ROC-AUC",
    ascending=False
)

display(results_df.round(3))


NameError: name 'preprocessor' is not defined

In [3]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Определяем числовые и категориальные столбцы.
# Важно: TotalCharges должен быть числовым столбцом.
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

# Числовые данные: пропуски заполняем медианой,
# затем масштабируем признаки, чтобы привести их к сопоставимому масштабу.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Категориальные данные: заполняем пропуски наиболее частым значением,
# затем преобразуем категории в числовые признаки.
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Объединяем обе обработки в один объект.
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

print("Числовые признаки:", len(numeric_cols))
print("Категориальные признаки:", len(categorical_cols))
print("Preprocessor создан.")


NameError: name 'X_train' is not defined

In [4]:

import pandas as pd

from sklearn.model_selection import train_test_split

# Загружаем исходный датасет.
df = pd.read_csv(
    "/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

# Преобразуем TotalCharges в числовой формат.
# Пустые строки станут NaN, чтобы их можно было обработать позже.
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Убираем идентификатор клиента и целевую переменную из признаков.
# TenureGroup тоже исключаем, если такой столбец существует.
X = df.drop(
    columns=["customerID", "Churn", "TenureGroup"],
    errors="ignore"
)

# Превращаем Yes/No в числа:
# 1 означает уход, 0 — клиент остался.
y = (df["Churn"] == "Yes").astype(int)

# Делим данные: 80% для обучения, 20% для проверки.
# stratify сохраняет примерно одинаковую долю ушедших клиентов в обеих выборках.
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Данные восстановлены.")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("Доля ушедших клиентов:", round(y.mean() * 100, 2), "%")


Данные восстановлены.
X_train: (5634, 19)
X_val: (1409, 19)
Доля ушедших клиентов: 26.54 %


In [5]:

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.base import clone

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# 1. Разделяем признаки по типу данных.
# Числовые признаки обрабатываем отдельно от текстовых категорий.
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

# 2. Подготовка числовых признаков:
# заполняем пропуски медианой и масштабируем значения.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 3. Подготовка категориальных признаков:
# заполняем пропуски самым частым значением и кодируем категории числами.
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# 4. Объединяем обе обработки в один объект.
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

# 5. Создаём три алгоритма с заданными настройками.
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
}

results = []

# 6. Обучаем и проверяем каждую модель одинаковым способом.
for name, classifier in models.items():

    # Для каждой модели создаём отдельную обработку признаков.
    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", classifier)
    ])

    # Модель учится только на обучающей выборке.
    pipeline.fit(X_train, y_train)

    # Предсказываем класс клиента и вероятность его ухода.
    predictions = pipeline.predict(X_val)
    probabilities = pipeline.predict_proba(X_val)[:, 1]

    # Сохраняем метрики для дальнейшего сравнения.
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, predictions),
        "Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_val, predictions, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_val, probabilities)
    })

# 7. Выводим таблицу от лучшего ROC-AUC к худшему.
results_df = pd.DataFrame(results).sort_values(
    by="ROC-AUC",
    ascending=False
)

display(results_df.round(3))


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.806,0.657,0.559,0.604,0.842
2,Random Forest,0.803,0.663,0.521,0.584,0.841
1,Decision Tree,0.798,0.635,0.567,0.599,0.830


In [1]:

# 1. Импортируем необходимые библиотеки.
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.base import clone

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# 2. Загружаем датасет.
df = pd.read_csv(
    "/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

# TotalCharges содержит пустые строки, поэтому преобразуем его в число.
# Пустые или некорректные значения станут NaN.
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

# 3. Отделяем признаки от целевой переменной.
# customerID — только идентификатор, он не нужен для прогноза.
# TenureGroup исключаем, если этот столбец был создан ранее.
X = df.drop(
    columns=["customerID", "Churn", "TenureGroup"],
    errors="ignore"
)

# Кодируем цель: 1 — клиент ушёл, 0 — остался.
y = (df["Churn"] == "Yes").astype(int)

# 4. Делим данные на обучение и проверку.
# stratify сохраняет похожую долю ушедших клиентов в обеих выборках.
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 5. Определяем типы признаков.
numeric_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

# 6. Настраиваем обработку числовых признаков.
# Медиана заполняет пропуски, StandardScaler масштабирует числа.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 7. Настраиваем обработку категорий.
# OneHotEncoder превращает текстовые категории в числовые признаки.
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Объединяем оба способа обработки.
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

print("Данные восстановлены:", df.shape)
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Доля оттока:", round(y.mean() * 100, 2), "%")

# 8. Сравниваем три алгоритма на одинаковых данных.
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
}

results = []

for name, classifier in models.items():
    # Каждая модель получает собственный экземпляр preprocessing.
    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", classifier)
    ])

    # Обучаем только на Train.
    pipeline.fit(X_train, y_train)

    # Проверяем качество на Validation.
    predictions = pipeline.predict(X_val)
    probabilities = pipeline.predict_proba(X_val)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, predictions),
        "Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "F1": f1_score(y_val, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_val, probabilities)
    })

results_df = pd.DataFrame(results).sort_values(
    "ROC-AUC", ascending=False
)

print("\nСравнение моделей:")
display(results_df.round(3))

# 9. Отдельно проверяем переобучение Random Forest.
rf_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

# Получаем прогнозы отдельно для обучающей и проверочной выборок.
train_pred = rf_model.predict(X_train)
val_pred = rf_model.predict(X_val)

train_prob = rf_model.predict_proba(X_train)[:, 1]
val_prob = rf_model.predict_proba(X_val)[:, 1]

# Большой разрыв между Train и Validation может указывать на overfitting.
overfitting_check = pd.DataFrame({
    "Dataset": ["Train", "Validation"],
    "Accuracy": [
        accuracy_score(y_train, train_pred),
        accuracy_score(y_val, val_pred)
    ],
    "F1": [
        f1_score(y_train, train_pred),
        f1_score(y_val, val_pred)
    ],
    "ROC-AUC": [
        roc_auc_score(y_train, train_prob),
        roc_auc_score(y_val, val_prob)
    ]
})

print("\nПроверка переобучения Random Forest:")
display(overfitting_check.round(3))


Данные восстановлены: (7043, 21)
Train: (5634, 19)
Validation: (1409, 19)
Доля оттока: 26.54 %

Сравнение моделей:


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.806,0.657,0.559,0.604,0.842
2,Random Forest,0.803,0.663,0.521,0.584,0.841
1,Decision Tree,0.798,0.635,0.567,0.599,0.830



Проверка переобучения Random Forest:


,Dataset,Accuracy,F1,ROC-AUC
0,Train,0.873,0.741,0.953
1,Validation,0.803,0.584,0.841


In [2]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
import pandas as pd

# Проверяем четыре варианта глубины дерева.
depths = [3, 5, 7, 10]
depth_results = []

for depth in depths:

    # Создаём Random Forest с выбранной глубиной.
    # Все остальные параметры оставляем одинаковыми.
    rf = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            max_depth=depth,
            random_state=42,
            n_jobs=-1
        ))
    ])

    # Обучаем модель на тренировочных данных.
    rf.fit(X_train, y_train)

    # Получаем прогнозы для обеих выборок.
    train_pred = rf.predict(X_train)
    val_pred = rf.predict(X_val)

    # Вероятности нужны для расчёта ROC-AUC.
    train_prob = rf.predict_proba(X_train)[:, 1]
    val_prob = rf.predict_proba(X_val)[:, 1]

    # Сохраняем метрики и разницу между выборками.
    depth_results.append({
        "Max depth": depth,
        "Train F1": f1_score(y_train, train_pred),
        "Validation F1": f1_score(y_val, val_pred),
        "Train ROC-AUC": roc_auc_score(y_train, train_prob),
        "Validation ROC-AUC": roc_auc_score(y_val, val_prob),
        "F1 gap": (
            f1_score(y_train, train_pred)
            - f1_score(y_val, val_pred)
        )
    })

# Сравниваем результаты: чем выше Validation F1, тем лучше баланс
# Precision и Recall при стандартном пороге классификации.
depth_df = pd.DataFrame(depth_results)
display(depth_df.round(3))


,Max depth,Train F1,Validation F1,Train ROC-AUC,Validation ROC-AUC,F1 gap
0,3,0.442,0.422,0.844,0.836,0.019
1,5,0.540,0.529,0.859,0.841,0.012
2,7,0.616,0.578,0.887,0.843,0.038
3,10,0.741,0.584,0.953,0.841,0.158


In [3]:

from sklearn.model_selection import StratifiedKFold, cross_validate

# Создаём 5 разбиений, сохраняя долю ушедших клиентов в каждом.
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = []

for depth in [7, 10]:

    # Создаём модель с заданной глубиной.
    # Обработка признаков заново обучается внутри каждого разбиения.
    rf = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            max_depth=depth,
            random_state=42,
            n_jobs=-1
        ))
    ])

    # Оцениваем качество на пяти проверочных частях.
    scores = cross_validate(
        rf,
        X_train,
        y_train,
        cv=cv,
        scoring=["f1", "roc_auc"],
        n_jobs=-1
    )

    # Сохраняем среднее качество и разброс результатов.
    cv_results.append({
        "Max depth": depth,
        "Mean F1": scores["test_f1"].mean(),
        "F1 std": scores["test_f1"].std(),
        "Mean ROC-AUC": scores["test_roc_auc"].mean(),
        "ROC-AUC std": scores["test_roc_auc"].std()
    })

display(pd.DataFrame(cv_results).round(3))


,Max depth,Mean F1,F1 std,Mean ROC-AUC,ROC-AUC std
0,7,0.568,0.018,0.847,0.01
1,10,0.579,0.021,0.842,0.01


In [4]:

from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np

# Создаём логистическую регрессию с той же подготовкой данных.
# Модель оценивает вероятность того, что клиент уйдёт.
log_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Обучаем модель на тренировочных данных.
log_model.fit(X_train, y_train)

# Получаем названия признаков после обработки:
# например, Contract_Two year или InternetService_Fiber optic.
feature_names = (
    log_model.named_steps["preprocessor"]
    .get_feature_names_out()
)

# Извлекаем коэффициенты обученной логистической регрессии.
coefficients = (
    log_model.named_steps["classifier"]
    .coef_[0]
)

# Собираем коэффициенты в таблицу.
feature_analysis = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Odds_Ratio": np.exp(coefficients)
})

# Сначала покажем признаки с наиболее положительными коэффициентами.
# Положительный коэффициент означает более высокие log-odds оттока
# при прочих равных условиях модели.
display(
    feature_analysis
    .sort_values("Coefficient", ascending=False)
    .head(10)
    .round(3)
)


,Feature,Coefficient,Odds_Ratio
16,cat__InternetService_Fiber optic,0.640,1.897
36,cat__Contract_Month-to-month,0.579,1.785
3,num__TotalCharges,0.517,1.676
35,cat__StreamingMovies_Yes,0.204,1.226
32,cat__StreamingTV_Yes,0.204,1.226
43,cat__PaymentMethod_Electronic check,0.198,1.219
18,cat__OnlineSecurity_No,0.157,1.170
27,cat__TechSupport_No,0.132,1.141
14,cat__MultipleLines_Yes,0.106,1.112
0,num__SeniorCitizen,0.054,1.055
